In [1]:
import cv2
import mediapipe as mp
import numpy as np

In [ ]:
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
Options = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = Options(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=1
)

cam = cv2.VideoCapture(0)
canvas = None
previous_point = None

with HandLandmarker.create_from_options(options) as hands:
    while True:
        ret, img = cam.read()
        if not ret:
            break

        img = cv2.flip(img, 1)
        h, w, _ = img.shape

        if canvas is None:
            canvas = np.zeros_like(img)

        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp = int(
            cv2.getTickCount() * 1000 / cv2.getTickFrequency()
        )
        result = hands.detect_for_video(mp_img, timestamp)

        if result.hand_landmarks:
            hand = result.hand_landmarks[0]

            
            tip = hand[8]
            x, y = int(tip.x * w), int(tip.y * h)

            if previous_point is not None:
                cv2.line(
                    canvas, previous_point, (x, y),
                    (237,174,192), 5
                )

            previous_point = (x, y)
            cv2.circle(img, (x, y), 8, (0, 0, 255), -3)

        else:
            previous_point = None

        # نمایش نقاشی روی تصویر دوربین
        output = cv2.addWeighted(img, 1, canvas, 1, 0)
        cv2.imshow("Finger Drawing", output)

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break
        elif key == ord('c'):
            canvas[:] = 0

cam.release()
cv2.destroyAllWindows()